# SFT data prep — Alpaca

Mounts Drive, downloads + tokenizes the SFT dataset, eyeballs 5 examples.
All real logic lives in `download_sft_data.py` and `tokenize_sft_data.py`.

In [ ]:
# Mount Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    SYNAPSE_DIR = '/content/drive/MyDrive/synapse'
except ImportError:
    import os
    SYNAPSE_DIR = os.environ.get('SYNAPSE_DIR') or os.path.abspath('./synapse')
import os
os.environ['SYNAPSE_DIR'] = SYNAPSE_DIR
print('SYNAPSE_DIR =', SYNAPSE_DIR)

In [ ]:
# Install deps (idempotent — skip if already present)
!pip install -q datasets tokenizers

In [ ]:
# Locate the synapse repo. Adjust if your clone is elsewhere.
import os, sys
REPO_DIR = '/content/synapse' if os.path.isdir('/content/synapse') else os.path.abspath('..')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('REPO_DIR =', REPO_DIR)

In [ ]:
# Step 1: download raw Alpaca → $SYNAPSE_DIR/datasets_sft/alpaca/
!cd {REPO_DIR} && SYNAPSE_DIR={SYNAPSE_DIR} python sft/download_sft_data.py --datasets alpaca

In [ ]:
# Step 2: tokenize → $SYNAPSE_DIR/sft_tokenized/alpaca/
!cd {REPO_DIR} && SYNAPSE_DIR={SYNAPSE_DIR} python sft/tokenize_sft_data.py --datasets alpaca --block-size 2048

In [ ]:
# Step 3: eyeball 5 tokenized examples — confirm prompt/response boundary and EOS
import json, random
from tokenizers import Tokenizer
tok = Tokenizer.from_file(os.path.join(SYNAPSE_DIR, 'tokenizer_out', 'tokenizer.json'))
EOT_ID = tok.token_to_id('<|endoftext|>')
IM_END_ID = tok.token_to_id('<|im_end|>')
lines = open(os.path.join(SYNAPSE_DIR, 'sft_tokenized', 'alpaca', 'train.jsonl')).readlines()
for i in random.sample(range(len(lines)), 5):
    ex = json.loads(lines[i])
    ids = ex['input_ids']
    pl = ex['prefix_len']
    print('=' * 60)
    print(f'idx={i}  total={len(ids)}  prefix_len={pl}')
    print('--- PROMPT (masked, not trained on) ---')
    print(tok.decode(ids[:pl]))
    print('--- RESPONSE (trained on) ---')
    print(tok.decode(ids[pl:]))
    print('--- last 3 tokens ---', ids[-3:], '|',
          [tok.id_to_token(t) for t in ids[-3:]])
    assert ids[-1] == EOT_ID, 'last token should be <|endoftext|>'
    assert ids[-2] == IM_END_ID, 'second-to-last should be <|im_end|>'
print('\nAll 5 examples passed EOS sanity checks.')